# 01 - Profile and chunk a real scientific corpus

**Objective:** audit 5,183 abstracts, preserve exact source offsets, and measure the consequences of chunking.
**Setup:** run `python scripts/prepare_data.py` once from the repository root. After that, this notebook runs offline using standard Python. Allow roughly a minute on a laptop; runtime depends on hardware.

We use the full [BEIR SciFact corpus](https://huggingface.co/datasets/BeIR/scifact), not a sample selected for easy retrieval. Queries are scientific claims and relevance labels identify useful evidence documents. A relevant document may support or refute a claim. See [dataset provenance and licensing](../../data/README.md).


In [ ]:
from pathlib import Path
import sys
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "rag_lab/core.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Start Jupyter inside the Awesome-RAG checkout")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from rag_lab.core import load_dataset, chunk_document, LexicalIndex, assemble_context
corpus, queries, qrels, manifest = load_dataset(ROOT)
print(f"{len(corpus):,} abstracts; {len(qrels['train']):,} development claims; {len(qrels['test']):,} test claims")
print("Snapshot:", manifest["archive_sha256"])


## Audit the corpus before choosing a chunk size

Check the distribution rather than relying on one example. The loader verifies file hashes, unique IDs, nonempty text, relevance references, split separation, and expected counts. Duplicate abstracts may still exist under different IDs; we measure those separately.


In [ ]:
from collections import Counter
from statistics import median
lengths = sorted(len(doc["text"].split()) for doc in corpus.values())
print({"minimum_words": lengths[0], "median_words": median(lengths),
       "p95_words": lengths[int(.95 * (len(lengths) - 1))], "maximum_words": lengths[-1]})
duplicates = Counter(doc["text"].strip() for doc in corpus.values())
print("Repeated abstract texts:", sum(count - 1 for count in duplicates.values()))
print("Missing titles:", sum(not doc["title"].strip() for doc in corpus.values()))
print("Judgments per split:", {name: sum(map(len, rows.values())) for name, rows in qrels.items()})


## Inspect the chunking implementation

Word windows are a reproducible baseline. They are not model tokens or sentence boundaries. Each chunk retains its parent document ID, title, and exact half-open character span. The shared helper is shown below so you can inspect the mechanism used by every notebook.


In [ ]:
import inspect
print(inspect.getsource(chunk_document))


In [ ]:
original_words = sum(lengths)
for size, overlap in [(80, 0), (120, 30), (240, 40)]:
    chunks = [chunk for doc in corpus.values() for chunk in chunk_document(doc, size, overlap)]
    stored_words = sum(len(chunk["text"].split()) for chunk in chunks)
    print({"size": size, "overlap": overlap, "chunks": len(chunks),
           "word_storage_multiplier": round(stored_words / original_words, 3)})
    for chunk in chunks:
        original = corpus[chunk["document_id"]]["text"]
        assert chunk["text"] == original[chunk["start_char"]:chunk["end_char"]]


## Inspect one development claim and its labeled source

Use the training split for exploration. The BEIR test split stays out of example selection. The source below is chosen from its label for inspection, not used as a retrieval shortcut.


In [ ]:
qid = sorted(qrels["train"])[0]
doc_id = sorted(qrels["train"][qid])[0]
print("CLAIM:", queries[qid]["text"])
print("SOURCE:", doc_id, corpus[doc_id]["title"])
for chunk in chunk_document(corpus[doc_id], 120, 30):
    print("\n", chunk["id"], chunk["text"])


## Exercise

Add a sentence-aware splitter and measure its chunk count and storage multiplier. Preserve exact source spans. Then compare retrieval using lesson 04's document-level labels: chunking has not improved anything merely because the passages look cleaner.

This corpus contains abstracts, not full PDFs. It does not test OCR, tables, layout, or full-document parsing. Next: [02 - Lexical retrieval](02_tfidf_retrieval.ipynb).
